> <p><small>This notebook is made available subject to the licence and terms set out in <a href="https://creativecommons.org/licenses/by/4.0">https://creativecommons.org/licenses/by/4.0</a>.</small></p>

<img src="https://pub-bba109a9a6ac49e3b428cdca19c34363.r2.dev/LT%20-%20Session%200.jpg">

# 0.6 AI Long Activity 1 - Lab: N-gram Name Generator (Teacher)

This teacher version contains a worked solution and facilitation notes for the Session 0 challenge lab. It supports teachers in explaining how simple letter-level statistics can generate new name-like outputs.

60 minutes

## Overview

In this lab, students build a simple character-level n-gram model for first names. The key aim is to help students understand that generative models can produce plausible outputs by learning statistical patterns from data.

### What students will learn:

- Inspect a small dataset of African first names.
- Build letter-level bigram counts.
- Generate new names using those counts.
- Vary randomness using temperature.
- Reflect on links between this simple model and larger language models.

> 💻 **Your tasks:**  
> Use this notebook to support student groups as they work through the lab.  
> The main teacher role is to help students connect code outputs to concepts:
> 1. Data contains patterns.
> 2. The model counts local transitions.
> 3. Sampling introduces randomness.
> 4. Generated outputs reflect both the dataset and the sampling process.

> ℹ️ **Info: Pedagogical focus**
>
> This is not intended to be a full programming lesson or a formal introduction to language modelling. The purpose is to give students an accessible first experience of how statistical patterns can generate text-like outputs.

## Step 1 – Load and inspect the dataset

Students begin by looking at the data before building the model. Encourage them to notice repeated letters, common beginnings, and common endings.

In [ ]:
# Load the dataset and set a random seed for reproducibility.
import random

random.seed(42)

african_names = [
    "Amina", "Kofi", "Ama", "Sibusiso", "Zanele", "Chinedu", "Ngozi",
    "Kwame", "Fatou", "Ibrahim", "Ayo", "Naledi", "Thabo", "Lerato",
    "Adeola", "Femi", "Zuri", "Jabari", "Akosua", "Nyasha"
]

print(f"Number of names in dataset: {len(african_names)}")
print("Example names:", african_names[:10])

> 💭 **Reflection – expected observations**
>
> Students may notice:
>
> - common endings such as `a`, `i`, or `o`
> - repeated letter pairs such as `na`, `ko`, or `mi`
> - names beginning with vowels or with letters such as `K`, `N`, `S`, or `T`
>
> Reinforce that these patterns come from the dataset. Different datasets would produce different patterns.

## Step 2 – Build bigram counts

This code builds a simple transition table. For each character, it counts which characters appear immediately after it.

> ℹ️ **Info: Start and end markers**
>
> The `^` and `$` markers are not letters in the names. They help the model learn how names start and stop.
>
> Without these markers, the generator would not have a simple way to know when to begin or end a generated name.

In [ ]:
# Build bigram transition counts from the dataset.
from collections import Counter, defaultdict


def build_bigram_counts(names):
    """Build bigram transition counts from a list of names.

    Input:
        names: list of strings

    Output:
        counts: dictionary mapping each character to counts of possible next characters
    """
    counts = defaultdict(Counter)

    for name in names:
        name_clean = name.strip()

        if not name_clean:
            continue

        # Add start and end markers.
        seq = "^" + name_clean + "$"

        # Count each pair of consecutive characters.
        for c1, c2 in zip(seq, seq[1:]):
            counts[c1][c2] += 1

    return counts


bigram_counts = build_bigram_counts(african_names)

## Step 3 – Inspect the learned counts

Use this output to show that the model has learned local transition statistics, not meaning.

In [ ]:
# Inspect next-letter counts after selected characters.
for first_char in ["^", "A", "K", "N"]:
    print(
        f"Next-letter counts after '{first_char}':",
        bigram_counts[first_char],
    )

> 💭 **Reflection – teacher prompts**
>
> Ask students:
>
> - Which letters commonly start names in this dataset?
> - Which transitions are frequent?
> - Which transitions are missing?
> - What might happen if the dataset were much larger or from a different region?

## Step 4 – Generate new names

The generator starts at `^`, repeatedly samples the next character, and stops at `$` or at a maximum length.

> ℹ️ **Info: Temperature**
>
> Temperature controls how strongly the generator favors common transitions.
>
> - Low temperature: safer, more repetitive outputs.
> - Medium temperature: balance between plausibility and variety.
> - High temperature: more varied, but often less plausible outputs.

In [ ]:
# Define sampling and generation functions.

def sample_next_letter(counter, temperature=1.0):
    """Sample the next letter from a weighted character distribution.

    Input:
        counter: counts of possible next characters
        temperature: controls randomness

    Output:
        one sampled character
    """
    items = list(counter.items())

    if not items:
        return "$"

    letters, counts = zip(*items)

    # Adjust counts using temperature.
    adjusted_counts = [count ** (1.0 / temperature) for count in counts]
    total = sum(adjusted_counts)
    probabilities = [count / total for count in adjusted_counts]

    return random.choices(
        letters,
        weights=probabilities,
        k=1,
    )[0]


def generate_name(bigram_counts, temperature=1.0, max_len=12):
    """Generate a name from bigram counts.

    Input:
        bigram_counts: learned character transition counts
        temperature: controls randomness
        max_len: maximum generated name length

    Output:
        generated name as a string
    """
    name = ""
    current = "^"

    while True:
        next_char = sample_next_letter(
            bigram_counts[current],
            temperature=temperature,
        )

        if next_char == "$" or len(name) >= max_len:
            break

        name += next_char
        current = next_char

    return name

## Step 5 – Experiment with temperature

Run the generator at several temperature values. Outputs vary because the sampling process is random.

In [ ]:
# Generate names at different temperatures.
for temp in [0.5, 1.0, 1.5]:
    print(f"
Temperature = {temp}")

    for _ in range(5):
        print(generate_name(bigram_counts, temperature=temp))

> 💭 **Reflection – suggested answers**
>
> 1. **How do generated names change when temperature changes?**
>    - Lower temperature usually gives more predictable and conservative outputs.
>    - Higher temperature usually gives more varied and sometimes unrealistic outputs.
>
> 2. **Which temperature gives the most plausible names?**
>    - Often around `1.0`, though this depends on the dataset and random sampling.
>
> 3. **Why might unrealistic names appear?**
>    - The model only uses local character transitions. It does not understand global linguistic structure, culture, or meaning.
>
> 4. **How does this relate to larger language models?**
>    - Larger language models use much longer context and many more parameters, but they also generate outputs by learning patterns from data and sampling possible next tokens.

## Optional extension – Use a local dataset

Teachers may prepare a text file containing one name per line, such as `class_names.txt`.

In [ ]:
# Optional local dataset loading example.
# Uncomment and adapt if a local file is available.

# with open("class_names.txt") as f:
#     class_names = [line.strip() for line in f if line.strip()]

# print(class_names[:10])
# class_bigram_counts = build_bigram_counts(class_names)

# for temp in [0.5, 1.0, 1.5]:
#     print(f"
Temperature = {temp}")
#     for _ in range(5):
#         print(generate_name(class_bigram_counts, temperature=temp))

> 💭 **Reflection – local dataset**
>
> Ask students:
>
> - Do generated names reflect the local dataset more closely?
> - Are there mixtures of naming styles or languages?
> - What does this show about the relationship between training data and model outputs?